In [ ]:
import os, sys
sys.path.insert(0, r"d:\Supply Chain Resilience")
# !"{sys.executable}" -m pip install pandera
# !"{sys.executable}" -m pip install scikit-learn

from src.data_loader import DataLoader
from src.data_cleaner import DataCleaner
from src.data_save import DataSave
from src.star_schema_builder import StarSchemaBuilder
import pandas as pd
from pathlib import Path
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check

# Capa bronce: ingesta de datos en crudo

In [2]:
ruta_bronce = Path("D:/Supply Chain Resilience/Dataset/01 Olist Bronce")
carga = DataLoader(ruta=str(ruta_bronce))
datos_bronze = carga.cargar_todo()

Cargando archivo olist_orders_dataset.csv
Cargando archivo olist_order_items_dataset.csv
Cargando archivo olist_customers_dataset.csv
Cargando archivo olist_products_dataset.csv
Cargando archivo olist_sellers_dataset.csv
Cargando archivo olist_order_payments_dataset.csv
Cargando archivo olist_order_reviews_dataset.csv
Cargando archivo olist_geolocation_dataset.csv
Se cargaron 8 datasets


In [3]:
datos_bronze.keys()

dict_keys(['Ordenes', 'Objetos_ordenes', 'clientes', 'productos', 'vendedores', 'pagos', 'reseñas', 'geolocalizacion'])

# Capa Silver: imputacion de valores, validacion de esquema y limpieza general

Vamos a crear el esquema solo para dos dataframes, pero podriamos definir mas esquemas de datos para los dataframe de base de datos


In [4]:
# esquema para el dataframe de orders
orders_schema = pa.DataFrameSchema({
    "order_id":     Column(str),
    "customer_id":  Column(str),
    "order_status": Column(str, Check.isin([
        "delivered", "shipped", "canceled",
        "unavailable", "invoiced","processing",
        "created","approved"
    ]))
})

# esquema para el dataframe de customers
customers_schema = pa.DataFrameSchema({
    "customer_unique_id":     Column(str),
    "customer_id":  Column(str),
    "customer_zip_code_prefix": Column(float),
    "customer_city": Column(str)
})

In [5]:
df_orders = (
    DataCleaner(datos_bronze["Ordenes"])
    .imputacion_num(n_neighbors=5)
    .validacion_esquema(orders_schema)
    .df_limpio()
)


No hay columnas numericas
Esquema validado correctamente


In [6]:
df_customers = (
    DataCleaner(datos_bronze["clientes"])
    .imputacion_num(n_neighbors=5)
    .validacion_esquema(customers_schema)
    .df_limpio()
)

Columnas imputadas: ['customer_zip_code_prefix']
Esquema validado correctamente


In [7]:
base_datos = {
    'df_customers'       : df_customers,
    'df_geolocation'     : datos_bronze['geolocalizacion'],
    'df_order_items'     : datos_bronze['Objetos_ordenes'],
    'df_order_payments'  : datos_bronze['pagos'],
    'df_order_reviews'   : datos_bronze['reseñas'],
    'df_orders'          : df_orders,
    'df_products'        : datos_bronze['productos'],
    'df_sellers'         : datos_bronze['vendedores']
}

Guardamos los datos en la carpeta 01 olist silver

In [8]:
ruta_silver = Path("D:/Supply Chain Resilience/Dataset/02 Olist Silver")
saver = DataSave(ruta=str(ruta_silver))
saver.guardar_todo(base_datos)


💾 Guardando Star Schema en Parquet...

✅ df_customers.parquet guardado | Tamaño: 6.69 MB
✅ df_geolocation.parquet guardado | Tamaño: 16.23 MB
✅ df_order_items.parquet guardado | Tamaño: 6.24 MB
✅ df_order_payments.parquet guardado | Tamaño: 3.66 MB
✅ df_order_reviews.parquet guardado | Tamaño: 9.16 MB
✅ df_orders.parquet guardado | Tamaño: 10.28 MB
✅ df_products.parquet guardado | Tamaño: 1.34 MB
✅ df_sellers.parquet guardado | Tamaño: 0.13 MB

🎉 Todos los archivos guardados exitosamente.


# Capa Gold: Consideramos solo los datos que seran usados para el analisis de datos

In [12]:

saver = DataSave(ruta=str(ruta_silver))
datos = saver.cargar_todo()

📂 df_customers.parquet cargado | Filas: 99,441
📂 df_geolocation.parquet cargado | Filas: 1,000,163
📂 df_orders.parquet cargado | Filas: 99,441
📂 df_order_items.parquet cargado | Filas: 112,650
📂 df_order_payments.parquet cargado | Filas: 103,886
📂 df_order_reviews.parquet cargado | Filas: 99,224
📂 df_products.parquet cargado | Filas: 32,951
📂 df_sellers.parquet cargado | Filas: 3,095


In [13]:
datos.keys()

dict_keys(['df_customers', 'df_geolocation', 'df_orders', 'df_order_items', 'df_order_payments', 'df_order_reviews', 'df_products', 'df_sellers'])

In [14]:
star = StarSchemaBuilder(
    df_customers       = datos['df_customers'],
    df_geolocation     = datos['df_geolocation'],
    df_order_items     = datos['df_order_items'],
    df_order_payments  = datos['df_order_payments'],
    df_order_reviews   = datos['df_order_reviews'],
    df_orders          = datos['df_orders'],
    df_products        = datos['df_products'],
    df_sellers         = datos['df_sellers']
)

base_datos = star.build_all()


🌟 Construyendo Star Schema...

✅ Fact Sales: (117604, 8)
✅ Dim customers: (99441, 4)
✅ Dim reviews: (99224, 4)
✅ Dim products: (32951, 4)
✅ Dim sellers: (3095, 4)
✅ Dim Time: (98875, 6)


In [15]:
ruta_gold = Path("D:/Supply Chain Resilience/Dataset/03 Olist Gold")
sv = DataSave(ruta=str(ruta_gold))
sv.guardar_todo(base_datos)


💾 Guardando Star Schema en Parquet...

✅ fact_sales.parquet guardado | Tamaño: 9.96 MB
✅ dim_geolocation.parquet guardado | Tamaño: 16.23 MB
✅ dim_order_items.parquet guardado | Tamaño: 6.24 MB
✅ dim_order_payments.parquet guardado | Tamaño: 3.66 MB
✅ dim_orders.parquet guardado | Tamaño: 10.28 MB
✅ dim_customers.parquet guardado | Tamaño: 3.59 MB
✅ dim_orders_reviews.parquet guardado | Tamaño: 7.94 MB
✅ dim_products.parquet guardado | Tamaño: 1.18 MB
✅ dim_sellers.parquet guardado | Tamaño: 0.13 MB
✅ dim_time.parquet guardado | Tamaño: 1.14 MB

🎉 Todos los archivos guardados exitosamente.
